# Notebook 00: System Architecture - DCN-v2 Ranking Pipeline

## Project: Multi-Model Hybrid Movie Recommendation with Deep Cross Network v2

### What This Project Changes vs. the Baseline

The baseline project is [two-tower-recsys](https://github.com/nbatra/two-tower-recsys), which uses XGBoost LambdaMART as the re-ranker. This project replaces that **XGBoost LambdaMART re-ranker** with **DCN-v2 (Deep & Cross Network v2)**, a neural ranking model purpose-built for learning feature interactions in recommendation systems. The retrieval stage (Two-Tower, ComiRec, SASRec) is unchanged.

| Component | Baseline ([two-tower-recsys](https://github.com/nbatra/two-tower-recsys)) | This Project |
|-----------|-----------------|---------------|
| Retrieval (candidate generation) | Two-Tower / ComiRec / SASRec | **Same** -- unchanged |
| FAISS indexing | IndexFlatIP, 128-dim | **Same** -- unchanged |
| Ranking (re-ranking) | XGBoost LambdaMART (tree ensemble) | **DCN-v2** (neural cross network + deep network) |
| Feature vector | 105 features (TT/SASRec) / 109 (ComiRec) | **Same** -- unchanged |
| Statistical evaluation | Basic metric comparison | **FDR-corrected paired tests, bootstrap CIs** |

### Why DCN-v2?

Google introduced DCN-v2 (2021, *DCN V2: Improved Deep & Cross Network*) to address a fundamental limitation of tree-based rankers: **XGBoost learns axis-aligned feature splits, not arbitrary feature interactions**.

Consider a ranking signal: "users who watch few movies AND rate them harshly AND prefer niche genres benefit more from high genre_match_score than mainstream users." XGBoost can learn this -- but only through deep trees with sequential splits on activity, rating_std, and genre_entropy before splitting on genre_match. Each path through the tree is independent, so the same interaction must be re-learned in many different tree contexts.

DCN-v2's Cross Network learns this as a single parameterized operation: it computes explicit bounded-degree polynomial interactions between ALL feature pairs in one forward pass, with shared parameters that generalize across users.

### Sections

1. The Full Pipeline (retrieval unchanged, ranking replaced)
2. DCN-v2 Architecture Deep Dive
3. DCN-v2 vs XGBoost: Theoretical Comparison
4. Feature Interaction Mechanics
5. Training Strategy: Loss Functions for Ranking
6. Production Deployment Considerations
7. Statistical Evaluation Framework (FDR, Bootstrap)
8. Notebook Dependency Graph

## Section 1: The Full Pipeline

```
+=============================================================================+
|                         OFFLINE PHASE (Training + Index Building)            |
+=============================================================================+
|
|  Step 1: Feature Engineering (Notebook 02) -- UNCHANGED
|  -----------------------------------------
|  Raw Data --> User Features (24-dim per user)
|          --> Item Features (73-dim per movie)
|          --> Cross Features (7-dim per user-movie pair)
|          --> Negative Samples (4 negatives per positive)
|          --> Sequential Interaction Histories (for ComiRec + SASRec)
|
|  Step 2: Retrieval Model Training (Notebooks 03 / 06 / 09) -- UNCHANGED
|  -----------------------------------------------------------
|  Two-Tower: user_features --> [User Tower] --> 128-dim, item_features --> [Item Tower] --> 128-dim
|  ComiRec:   user_sequence --> [Capsule Net] --> 4 x 128-dim
|  SASRec:    user_sequence --> [Transformer] --> 128-dim
|
|  Step 3: FAISS Index Building -- UNCHANGED
|  Store 21,082 item embeddings per retrieval model for ANN search.
|
|  Step 4: DCN-v2 Ranker Training (Notebooks 04 / 07 / 10) -- NEW
|  -----------------------------------------------------------
|  For each (user, movie, label) in training set:
|    features = retrieval_score + user_features(24) + item_features(73)
|             + cross_features(7) = 105 or 109 dim
|
|    Input: x_0 = features (105 or 109 dim)
|                |
|                +--> Cross Network (3 layers)
|                |      x_{l+1} = x_0 * (W_l @ x_l + b_l) + x_l
|                |      Output: 105-dim (same as input)
|                |
|                +--> Deep Network (MLP)
|                |      Linear(105, 256) -> ReLU -> BN -> Dropout
|                |      Linear(256, 128) -> ReLU -> BN -> Dropout
|                |      Output: 128-dim
|                |
|                +--> Concat(cross_out, deep_out) = 233-dim
|                +--> Linear(233, 1) -> score
|
|    Loss: Binary Cross-Entropy with Logits (pointwise)
|          + Pairwise BPR loss (within-group ranking signal)
|
|  Output: dcn_ranker.pt per retrieval model
|
+=============================================================================+
```

```
+=============================================================================+
|                         ONLINE PHASE (Inference) -- RANKING CHANGED          |
+=============================================================================+
|
|  Stage A: Model Selection (unchanged)
|    Route user to retrieval model based on profile
|
|  Stage B: Candidate Generation (unchanged)
|    FAISS.search(user_embedding, K=200) --> top-200 candidates
|
|  Stage C: Ranking -- CHANGED
|    1. Look up user_features (24-dim) from Feature KV
|    2. Look up item_features (73-dim) for each candidate
|    3. Compute cross_features (7-dim) and retrieval_score on the fly
|    4. Concatenate into 105/109-dim feature vector per candidate
|    5. DCN-v2 forward pass (no GPU needed -- small model):
|       - Cross Network: 3 matrix multiplications (105x105 each)
|       - Deep Network: 2 matrix multiplications (105x256, 256x128)
|       - Output projection: 1 matrix multiplication (233x1)
|    6. Scores = model output (logits)
|
|  Stage D: Post-Processing (unchanged)
|    Filter already-seen, MMR diversity re-ranking, return top-10
|
+=============================================================================+
```

## Section 2: DCN-v2 Architecture Deep Dive

### Origin and Motivation

DCN-v2 was published by Google Research in 2021 (*DCN V2: Improved Deep & Cross Network and Practical Lessons for Web-scale Learning to Rank Systems*, Ruoxi Wang et al.). It evolved from DCN (2017) which was deployed in Google Play's recommendation system.

The key insight: recommendation ranking requires learning **feature crosses** -- combinations of features that are predictive only when considered together. For example:
- `genre_match` alone: moderately predictive
- `genre_match` AND `user_is_selective_rater`: highly predictive (selective users care more about genre alignment)
- `genre_match` AND `user_is_selective_rater` AND `item_is_niche`: strongest signal (selective users seeking niche content in their preferred genre)

### Architecture Components

```
Input: x_0 in R^d (d = 105 features)

=== CROSS NETWORK (explicit feature interactions) ===

Layer 1: x_1 = x_0 * (W_1 @ x_0 + b_1) + x_0
         ^         ^                       ^
         |         |                       |
         |         Learned projection      Residual connection
         Element-wise product with input

Layer 2: x_2 = x_0 * (W_2 @ x_1 + b_2) + x_1
Layer 3: x_3 = x_0 * (W_3 @ x_2 + b_3) + x_2

After L layers: x_L captures up to (L+1)-degree feature interactions
  - 1 cross layer: learns pairwise interactions (feature_i * feature_j)
  - 2 cross layers: learns 3-way interactions (feature_i * feature_j * feature_k)
  - 3 cross layers: learns 4-way interactions

Output: x_cross in R^d (same dimensionality as input -- 105)

=== DEEP NETWORK (implicit high-order patterns) ===

h_1 = ReLU(BatchNorm(Linear(x_0, 256)))
h_2 = ReLU(BatchNorm(Linear(h_1, 128)))

Output: x_deep in R^128

=== COMBINATION ===

concat = [x_cross; x_deep] in R^(105 + 128) = R^233
score = Linear(concat, 1)  --> scalar logit
```

### DCN-v2 vs DCN-v1: What Changed

| Aspect | DCN-v1 (2017) | DCN-v2 (2021) |
|--------|--------------|---------------|
| Cross layer weight | Rank-1 matrix (w * w^T) | Full-rank matrix W (d x d) |
| Expressiveness | Limited to rank-1 interactions | Arbitrary feature interactions |
| Stacked vs Parallel | Stacked only | Stacked OR Parallel (we use parallel) |
| Practical impact | Often underfits | Matches or beats deep models |

DCN-v1's cross layer used a rank-1 projection (outer product of two vectors), which severely limited what interactions it could learn. DCN-v2 uses a full (d x d) weight matrix per cross layer, allowing it to learn arbitrary pairwise feature transformations.

### Why Parallel Architecture (not Stacked)

DCN-v2 offers two modes:
- **Stacked**: Input -> Cross Network -> Deep Network -> Output
- **Parallel**: Input -> [Cross Network || Deep Network] -> Concat -> Output

We use **Parallel** because:
1. Cross Network and Deep Network learn complementary representations (explicit vs implicit)
2. Gradients flow through both paths independently (no vanishing gradient through the full stack)
3. Google's paper reports parallel matches stacked on most benchmarks with better training stability
4. Parallel is easier to ablate: we can measure the contribution of each component independently

### Parameter Count

For our 105-feature input:

| Component | Parameters | Calculation |
|-----------|-----------|-------------|
| Cross Layer 1 | 105 x 105 + 105 = 11,130 | W_1 matrix + b_1 bias |
| Cross Layer 2 | 11,130 | Same |
| Cross Layer 3 | 11,130 | Same |
| Deep Layer 1 | 105 x 256 + 256 = 27,136 | Weight + bias |
| Deep BN 1 | 512 | gamma + beta |
| Deep Layer 2 | 256 x 128 + 128 = 32,896 | Weight + bias |
| Deep BN 2 | 256 | gamma + beta |
| Output Layer | 233 x 1 + 1 = 234 | Projection to scalar |
| **Total** | **~94,424** | Tiny model -- fast inference |

Compare to XGBoost: 175 trees x max_depth 8 = ~50K leaf nodes, each storing a weight. Comparable parameter count, fundamentally different inductive bias.

## Section 3: DCN-v2 vs XGBoost - Theoretical Comparison

### Feature Interaction Learning

| Aspect | XGBoost (tree ensemble) | DCN-v2 (cross network) |
|--------|------------------------|------------------------|
| Interaction type | Axis-aligned splits (if feature_i > threshold, go left) | Weighted polynomial crosses (feature_i * W * feature_j) |
| Interaction order | Depth-limited (max_depth=8 -> up to 8-way) | Layer-limited (3 layers -> up to 4-way) |
| Sharing | Each tree learns independently | Cross weights shared across all examples |
| Continuous features | Discretized at split points | Used directly as continuous values |
| Generalization | Memorizes observed split thresholds | Generalizes to unseen feature value combinations |
| Cold-start items | Poor (no leaf statistics) | Better (continuous feature projection) |

### Concrete Example

Suppose the true ranking signal is: `score = 0.3 * genre_match * log_activity + 0.7 * retrieval_score * (1 - popularity_gap)`

**XGBoost** must approximate this with piecewise-constant regions:
```
Tree 1: if genre_match > 0.5 AND log_activity > 0.3 -> leaf_value = 0.8
         if genre_match > 0.5 AND log_activity <= 0.3 -> leaf_value = 0.4
         if genre_match <= 0.5 -> leaf_value = 0.1
Tree 2: if retrieval_score > 0.7 AND popularity_gap < 0.2 -> leaf_value = 0.9
         ...
```
It needs many trees with many splits to approximate the smooth interaction surface.

**DCN-v2** learns this directly in one cross layer:
```
W[genre_match_row, log_activity_col] = 0.3  (learns the interaction coefficient)
W[retrieval_row, pop_gap_col] = -0.7         (learns negative interaction)
```
One matrix multiplication captures both interactions exactly.

### When XGBoost Wins

- **Sparse, irregular decision boundaries**: "If the movie was released between 1990-1995 AND has >1000 ratings AND is genre Horror -> boost." XGBoost finds these hard thresholds naturally.
- **Small datasets**: <100K training examples. XGBoost's leaf statistics need fewer samples to be reliable than neural gradients.
- **No GPU, strict latency**: XGBoost tree traversal is if/else. Neural networks require matrix multiplication.
- **Interpretability required**: Feature importance and SHAP values are well-defined for trees.

### When DCN-v2 Wins

- **Smooth feature interactions**: Retrieval_score * genre_match is a continuous surface, not a step function.
- **Large datasets**: >1M training examples. Neural networks improve with more data; XGBoost saturates.
- **High-dimensional input**: >100 features with many potential interactions. XGBoost's greedy split selection may miss important pairs.
- **Transfer learning**: Pre-trained DCN layers can be fine-tuned for new domains. XGBoost starts from scratch.
- **End-to-end optimization**: DCN-v2 gradients flow back to embedding layers if desired (joint retrieval+ranking training).

### Our Dataset: Which Should Win?

Our ranking dataset has:
- 3M training examples (moderate -- advantage: neural)
- 105 features (moderate dimensionality -- neutral)
- Mix of smooth (retrieval_score, genre_match) and discrete (genre indicators) features
- Temporal shift between train/val/test (robustness matters)

Expected outcome: DCN-v2 should match or slightly beat XGBoost on smooth interactions, while XGBoost may retain advantages on discrete boundary detection. The comparison in Notebook 05 will quantify this with proper statistical rigor.

## Section 4: Feature Interaction Mechanics

### What the Cross Network Actually Computes

Let x_0 = [retrieval_score, log_activity, avg_rating, ..., genre_match, pop_gap] (105 features).

**Cross Layer 1** computes:
```
x_1[i] = x_0[i] * sum_j(W_1[i,j] * x_0[j] + b_1[i]) + x_0[i]
```

Expanding for a specific feature, say `retrieval_score` (index 0):
```
x_1[0] = retrieval_score * (W[0,0]*retrieval + W[0,1]*activity + W[0,2]*avg_rating + ... + W[0,104]*hour_cos + b[0])
         + retrieval_score
```

This produces ALL pairwise interactions of retrieval_score with every other feature:
- retrieval_score * log_activity (scaled by W[0,1])
- retrieval_score * genre_match (scaled by W[0,98])
- retrieval_score * popularity_gap (scaled by W[0,99])
- ... and so on for all 105 features

After Layer 1: all 105 * 105 = 11,025 pairwise interactions exist in x_1.

**Cross Layer 2** takes x_1 (which already contains pairwise terms) and multiplies by x_0 again:
```
x_2[i] = x_0[i] * (W_2 @ x_1 + b_2)[i] + x_1[i]
```

This creates 3-way interactions:
- retrieval_score * genre_match * user_activity
- retrieval_score * popularity_gap * avg_rating
- etc.

**Cross Layer 3** adds 4-way interactions. For our 105 features, the space of possible 4-way interactions is C(105,4) = ~4.8M. The cross network learns which of these are useful via the weight matrices.

### Comparison with Manual Feature Engineering

In the XGBoost pipeline, we manually created 7 cross-features:
```
genre_match_score    = dot(user_genre_prefs, item_genres)
popularity_gap       = item_popularity - user_avg_popularity
movie_age_at_rating  = rating_year - release_year
dow_sin, dow_cos     = cyclic encoding of day-of-week
hour_sin, hour_cos   = cyclic encoding of hour
```

These 7 features capture interactions we **know** are important. But there are 11,025 possible pairwise interactions and ~190K possible 3-way interactions. We cannot manually engineer all of them. DCN-v2 learns which interactions matter from data.

### Interactions DCN-v2 Might Discover That We Missed

- `retrieval_score * item_avg_rating * user_rating_std`: How much should we trust the retrieval score given the item's general quality and the user's selectiveness?
- `genome_pca_0 * user_pref_SciFi * movie_age`: PCA component interactions with genre preferences and recency
- `genre_match * log_activity * popularity_gap`: Genre alignment matters differently for active users browsing niche content vs. casual viewers

These are the interactions that XGBoost approximates through deep tree paths but cannot parameterize efficiently.

## Section 5: Training Strategy

### Loss Functions for Neural Ranking

Unlike XGBoost's built-in `rank:ndcg` objective which directly optimizes NDCG via LambdaMART gradients, neural rankers require explicit loss function choices:

| Loss | Type | Formula | Pros | Cons |
|------|------|---------|------|------|
| BCE | Pointwise | -y*log(p) - (1-y)*log(1-p) | Simple, well-calibrated scores | Ignores relative ordering |
| BPR | Pairwise | -log(sigmoid(s_pos - s_neg)) | Directly optimizes ranking | Requires sampling pairs |
| ListNet | Listwise | KL(softmax(y), softmax(s)) | Considers full list context | Expensive, needs groups |
| ApproxNDCG | Listwise | Differentiable NDCG approximation | Closest to evaluation metric | Complex implementation |

### Our Strategy: Combined BCE + BPR

We use a **combined loss**:
```
L = alpha * BCE(scores, labels) + (1 - alpha) * BPR(pos_scores, neg_scores)
```

- **BCE** provides calibrated probability estimates (useful for confidence thresholds in production)
- **BPR** provides ranking signal (the model learns that positive items should score higher than negatives)
- alpha = 0.5 gives equal weight to both objectives

### Why Not Pure Listwise?

Listwise losses (ListNet, ApproxNDCG) consider the full candidate list for each user. This is theoretically optimal but practically challenging:
1. Our groups have variable sizes (1 to 8,581 candidates per user)
2. Large groups cause memory issues during backprop (softmax over 8K items)
3. BCE + BPR achieves >95% of listwise performance with simpler implementation and stable gradients

### Training Hyperparameters

| Parameter | Value | Rationale |
|-----------|-------|----------|
| Learning rate | 1e-3 | Standard for small models, with OneCycleLR schedule |
| Batch size | 4096 | Large enough for stable BPR pair sampling |
| Epochs | 30 (early stopping patience=5) | Monitor val NDCG@10 |
| Weight decay | 1e-5 | Light L2 regularization on all parameters |
| Dropout | 0.1 | Applied in Deep Network only (not Cross Network) |
| Cross layers | 3 | Up to 4-way feature interactions |
| Deep hidden dims | [256, 128] | Standard MLP capacity |
| Gradient clipping | max_norm=1.0 | Prevents exploding gradients from BPR loss |

### Learning Rate Schedule

OneCycleLR with:
- Warmup: 10% of total steps (linear ramp from lr/25 to lr)
- Cosine decay: remaining 90% of steps
- Final LR: lr/1000

The warmup prevents early instability from random cross-layer weights (which produce large feature interaction values before calibration).

## Section 6: Production Deployment Considerations

### Inference Latency

For 200 candidates (batch inference):

| Operation | XGBoost | DCN-v2 |
|-----------|---------|--------|
| Core computation | 175 tree traversals per candidate | 6 matrix multiplications (batched) |
| Parallelism | Per-tree (CPU SIMD) | Per-candidate (batch matmul) |
| Memory access pattern | Random (tree node jumps) | Sequential (matrix rows) |
| Expected latency (200 items, CPU) | ~0.6ms | ~0.5-1.0ms |
| GPU benefit | None (tree ops not GPU-friendly) | Moderate (small matrices, overhead dominates) |

DCN-v2 with 94K parameters and 200 candidates is well within CPU inference budget. No GPU needed at serving time.

### Model Serving Architecture

```
Option A: PyTorch CPU inference (what we use)
  - torch.no_grad() + model.eval()
  - Batch all 200 candidates in one forward pass
  - Latency: ~0.5-1.0ms

Option B: ONNX Runtime (production optimization)
  - Export model to ONNX format
  - ONNX Runtime applies graph optimizations (operator fusion, constant folding)
  - Latency: ~0.3-0.5ms (30-50% faster than raw PyTorch)

Option C: TorchScript (deployment without Python)
  - torch.jit.trace() for static graph export
  - Deploy in C++ inference server
  - Latency: ~0.3ms
```

### Model Update Strategy

| Aspect | XGBoost | DCN-v2 |
|--------|---------|--------|
| Incremental training | Not natively supported (retrain from scratch) | Fine-tune with new data (continue optimizer state) |
| A/B rollout | Swap model file | Swap model file (identical process) |
| Rollback | Load previous .json | Load previous .pt |
| Model size on disk | ~5 MB (175 trees, JSON) | ~0.4 MB (94K float32 params) |
| Warm-up needed | No (tree traversal is stateless) | First forward pass JIT-compiles (add 50ms warmup) |

### Monitoring Differences

XGBoost scores are unbounded (sum of leaf values). DCN-v2 output logits are also unbounded, but we can apply sigmoid for calibrated probabilities:

```python
# Production scoring
logits = dcn_model(feature_tensor)        # raw DCN-v2 output
probabilities = torch.sigmoid(logits)      # calibrated P(click)
rankings = torch.argsort(logits, dim=-1, descending=True)  # final ranking
```

This gives us **calibrated confidence scores** for free -- something XGBoost's ranking output cannot provide (its scores are relative, not probabilistic).

## Section 7: Statistical Evaluation Framework

### Why Rigorous Statistics Matter

The baseline project compared models by computing mean metrics and declaring a winner. This is insufficient for production decisions because:

1. **Random variation**: A difference of NDCG +0.003 might be noise from the specific test set sample
2. **Multiple comparisons**: Testing DCN-v2 vs XGBoost on 6 metrics simultaneously inflates false positive risk
3. **Effect heterogeneity**: A model might win on average but lose for critical user cohorts

---

### What Actually Happened in This Project

We evaluated DCN-v2 vs XGBoost across all three retrieval pipelines. Here are the actual results:

| Pipeline | NDCG@10 (DCN-v2) | NDCG@10 (XGBoost) | Relative Delta | Significant After FDR? |
|----------|------------------:|-------------------:|---------------:|:----------------------:|
| Two-Tower | 0.0502 | 0.0495 | +1.3% | No (0 / 9 metrics) |
| ComiRec | 0.0562 | 0.0593 | -5.3% | No (0 / 6 metrics) |
| SASRec | 0.0570 | 0.0574 | -0.7% | No (0 / 6 metrics) |

**Not a single metric was statistically significant after FDR correction.** The observed differences -- both positive and negative -- are indistinguishable from random noise given the test set size.

---

### FDR Correction Explained: Why We Cannot Trust Raw P-Values

#### The problem in plain language

Suppose you are a hiring manager and you test 6 candidates on a coding challenge. Each test has a 5% false positive rate (5% chance of passing a candidate who would actually perform poorly on the job). If you test just one candidate, that 5% risk is fine. But you tested six:

- Probability that **at least one** unqualified candidate passes by luck = 1 - (0.95)^6 = **26.5%**

Now map this to our evaluation. We tested DCN-v2 vs XGBoost on 6 metrics (NDCG@10, Hit@10, MRR@10, MAP@10, Precision@10, Recall@10) for each pipeline. Each individual test uses alpha = 0.05 (5% false positive threshold). But with 6 tests per pipeline:

- We have a 26.5% chance of finding **at least one "significant" result per pipeline even if both models are truly identical**
- Across all 3 pipelines (21 total tests): 1 - (0.95)^21 = **66% chance** of at least one false positive

Without correction, running enough tests virtually guarantees you will find something "significant" to report. This is exactly how papers and blog posts mislead: test many metrics, report only the one that passed p < 0.05, ignore the others.

#### What Benjamini-Hochberg FDR correction does

Instead of asking "is this one test significant at 5%?", it asks: "among ALL the tests I declare significant, what fraction are expected to be false positives?"

The procedure for our ComiRec pipeline (6 metrics):

1. Run Wilcoxon signed-rank test on each metric's paired differences
2. Collect all 6 raw p-values
3. Rank them smallest to largest
4. For each rank k, compare p_(k) against an adjusted threshold: (k / 6) * 0.05

```
Rank    Metric         Raw p-value    BH threshold       Pass?
1       recall@10      0.042          1/6 * 0.05 = 0.008   No (0.042 > 0.008)
2       hit@10         0.057          2/6 * 0.05 = 0.017   No
3       map@10         0.089          3/6 * 0.05 = 0.025   No
4       ndcg@10        0.112          4/6 * 0.05 = 0.033   No
5       mrr@10         0.134          5/6 * 0.05 = 0.042   No
6       precision@10   0.485          6/6 * 0.05 = 0.050   No
```

The key insight: **the smallest p-value (0.042 for recall@10) would pass a naive alpha=0.05 threshold.** Without correction, we might report "XGBoost significantly outperforms DCN-v2 on Recall@10 (p=0.042)." But BH recognizes this is the best result out of 6 attempts. The adjusted threshold for the smallest p-value is 0.05/6 = 0.008, not 0.05. Since 0.042 > 0.008, it correctly fails.

Think of it like this: if you roll a die 6 times, getting at least one "6" is not surprising (probability 67%). BH is asking "did you get a result so extreme that it would be surprising even given that you had 6 chances?"

#### Why this matters for deployment

Without FDR correction, our results could be misinterpreted:
- "DCN-v2 significantly improves Hit@10 on SASRec (p=0.038) -- ship it!"
- "XGBoost significantly beats DCN-v2 on Recall@10 for ComiRec (p=0.042) -- stay with XGBoost!"

Both statements are likely false positives from testing many metrics. With FDR correction, we get a clearer and more honest conclusion: **we have no statistical evidence that either model is meaningfully better on this dataset.**

This is a valid and useful finding. It tells us:
1. Do not incur engineering cost switching to DCN-v2 unless there are other benefits (calibrated outputs, fine-tuning flexibility)
2. The 105/109-dim feature set may lack sufficient interaction complexity for DCN-v2 to demonstrate its theoretical advantage over gradient-boosted trees
3. Larger-scale data (100M+ interactions, as in Google's original paper) is likely needed for DCN-v2 to clearly separate from XGBoost

---

### Our Full Evaluation Protocol (Technical Details)

**1. Paired Testing (per-user comparison)**

For each test user, both rankers score the *same* FAISS candidates. This gives paired observations:
```
delta_i = NDCG_DCN(user_i) - NDCG_XGB(user_i)
```

We test H0: mean(delta) = 0 using **Wilcoxon signed-rank** (non-parametric, no normality assumption on the deltas).

Why paired? Some users have 50 test positives (easy to get hits in top-10) while others have 2 (nearly impossible). By comparing both rankers on the *same* user, we remove this confound. The test is about whether DCN-v2 consistently ranks better than XGBoost *for the same user*, not whether some users are inherently easier.

**2. Bootstrap Confidence Intervals (10,000 resamples)**

```python
for b in range(10000):
    resample = np.random.choice(deltas, size=len(deltas), replace=True)
    bootstrap_means[b] = resample.mean()

CI_95 = [percentile(bootstrap_means, 2.5), percentile(bootstrap_means, 97.5)]
```

If the 95% CI for the mean difference excludes 0, we have evidence of a real effect. In our results, all CIs included 0 -- consistent with no true difference between models.

**3. Benjamini-Hochberg FDR Correction**

Applied across all metrics within each pipeline. Controls the expected false discovery proportion at 5%. See the detailed walkthrough above.

**4. Effect Size (Cohen's d)**

Even if a test passed FDR correction, we would check practical significance:
- `|d|` < 0.2: negligible (not worth deploying even if p < 0.001)
- 0.2 <= `|d|` < 0.5: small effect (marginal business case)
- 0.5 <= `|d|` < 0.8: medium effect (clear signal to deploy)
- `|d|` >= 0.8: large effect (unambiguous winner)

In our results, all Cohen's d values were below 0.1 (deep in "negligible" territory). Even if FDR correction had passed some tests, the effect sizes tell us the practical impact is too small to justify infrastructure changes.

### Decision Framework Applied to Our Results

```
Is the improvement statistically significant (FDR-adjusted p < 0.05)?
  ==> NO for all metrics across all pipelines
  ==> DECISION: Do not deploy DCN-v2 as a replacement for XGBoost
      based on ranking quality alone.

If the answer had been YES, the next check would be:
  Is the effect size meaningful (Cohen's d >= 0.2)?
    YES --> Is latency acceptable (P95 < SLA)?
      YES --> Deploy with A/B test monitoring (Notebook 13)
```

Our evaluation stops at the first gate. There is no statistically significant evidence that DCN-v2 improves ranking quality over XGBoost LambdaMART on the MovieLens-25M dataset with our 105/109-dimensional feature vectors.


## Section 8: Notebook Dependency Graph

```
Notebook 00 (this file) - Architecture Overview
  Produces: Nothing (reference document)
  Consumes: Nothing

Notebook 01 (EDA) -- UNCHANGED
  Produces: Insights and design decisions
  Consumes: Raw data (ml-25m/*.csv)

Notebook 02 (Feature Engineering) -- UNCHANGED
  Produces: item_features.parquet, user_features.parquet, train/val/test sets,
            interaction_features, metadata.pkl
  Consumes: Raw data (ml-25m/*.csv)

---------- TWO-TOWER PIPELINE ----------

Notebook 03 (Two-Tower Model Training) -- UNCHANGED
  Produces: two_tower_model.pt, faiss_index_128dim.bin,
            item/user_embeddings_128dim.npy
  Consumes: two_tower_train.parquet, features, metadata

Notebook 04 (DCN-v2 Ranking) -- NEW
  Produces:
    models/dcn_v2_ranker.pt              (trained DCN-v2 model)
    models/dcn_v2_training_history.pkl   (loss curves, metrics per epoch)
    models/dcn_v2_config.json            (architecture hyperparameters)
  Also loads and evaluates:
    models/xgboost_ranker.json           (baseline comparison)
  Consumes: train/val sets, features, NB03 embeddings

Notebook 05 (Evaluation with Statistical Rigor) -- NEW
  Produces: Metric reports with CIs, FDR-corrected comparisons,
            per-cohort analysis, calibration plots
  Consumes: NB03 + NB04 models (both DCN-v2 and XGBoost), test set

---------- COMIREC PIPELINE ----------

Notebook 06 (ComiRec Model Training) -- UNCHANGED
  Produces: comirec_model.pt, faiss_index.bin,
            item/user_embeddings.npy (multi-interest)
  Consumes: Sequential data, features, metadata

Notebook 07 (ComiRec DCN-v2 Ranking) -- NEW
  Produces:
    models/comirec/dcn_v2_ranker.pt
    models/comirec/dcn_v2_training_history.pkl
  Consumes: train/val sets, features, NB06 embeddings

Notebook 08 (ComiRec Evaluation) -- UPDATED
  Produces: Two-way comparison (XGBoost vs DCN-v2) for ComiRec retrieval
  Consumes: NB06 + NB07 models

---------- SASREC PIPELINE ----------

Notebook 09 (SASRec Model Training) -- UNCHANGED
  Produces: sasrec_model.pt, faiss_index.bin,
            item/user_embeddings.npy
  Consumes: Sequential data, features, metadata

Notebook 10 (SASRec DCN-v2 Ranking) -- NEW
  Produces:
    models/sasrec/dcn_v2_ranker.pt
    models/sasrec/dcn_v2_training_history.pkl
  Consumes: train/val sets, features, NB09 embeddings

Notebook 11 (Three-Way Evaluation) -- UPDATED
  Produces: Full comparison across retrieval models x ranker models
  Consumes: All models

---------- PRODUCTION ----------

Notebook 12 (Production Inference Simulation) -- UPDATED
  Produces: Latency benchmarks, DCN-v2 vs XGBoost serving comparison
  Consumes: All models + features

Notebook 13 (A/B Testing Framework) -- UPDATED
  Produces: Sequential testing with alpha-spending,
            FDR-corrected multi-metric decisions
  Consumes: All models + test ground truth
```

### Data flow diagram:

```
Raw Data (ml-25m)
    |
    v
NB02 (Feature Engineering)
    |
    +--> NB03 (Two-Tower) --> NB04 (DCN-v2 + XGBoost comparison) --> NB05 (Eval)
    |                                                                    |
    +--> NB06 (ComiRec)  --> NB07 (DCN-v2 + XGBoost comparison) --> NB08 (Eval)
    |                                                                    |
    +--> NB09 (SASRec)   --> NB10 (DCN-v2 + XGBoost comparison) --> NB11 (Eval)
    |                                                                    |
    +--------------------------------------------------------------------+
    |                                                                    |
    v                                                                    v
NB12 (Production Service with DCN-v2)                    NB13 (A/B Testing)
```

## Section 9: Key Differences from Baseline Project

Baseline repository: [https://github.com/nbatra/two-tower-recsys](https://github.com/nbatra/two-tower-recsys)

### What is NOT changing

1. Raw data and feature engineering (NB01, NB02)
2. All three retrieval models (Two-Tower, ComiRec, SASRec)
3. FAISS indexing and candidate generation
4. Feature vector structure (105/109 dimensions)
5. The 7 cross-features (genre_match, popularity_gap, temporal)
6. Post-processing (MMR diversity, already-seen filtering)

### What IS changing

1. **Ranking model**: XGBoost LambdaMART -> DCN-v2
2. **Loss function**: rank:ndcg (LambdaMART) -> BCE + BPR (combined)
3. **Inference**: `xgb.Booster.predict()` -> `torch.no_grad(): model(tensor)`
4. **Model artifact**: `.json` tree ensemble -> `.pt` neural network state dict
5. **Feature importance**: XGBoost gain-based -> Gradient-based attribution / ablation
6. **Statistical evaluation**: Mean comparison -> FDR-corrected paired tests with bootstrap CIs
7. **Calibration**: XGBoost scores (relative) -> DCN-v2 sigmoid outputs (probabilistic)

### Deeper Rationale for Each Change

**Ranking model swap (XGBoost -> DCN-v2):** The core hypothesis of this project is that explicit polynomial feature crosses -- as parameterized by DCN-v2's cross layers -- can capture interaction signals that XGBoost's axis-aligned splits approximate only through deep, redundant tree paths. With 105 input features, the space of possible pairwise interactions is 5,460 (C(105,2)), and XGBoost must re-discover each relevant pair independently within every tree. DCN-v2's weight matrices learn these interactions as shared parameters once, generalizing across the full training distribution.

**Loss function swap (LambdaMART -> BCE + BPR):** LambdaMART computes gradients that directly approximate changes in NDCG when swapping two items in the ranked list -- this is elegant but tightly coupled to the specific tree ensemble optimization. For neural networks, BCE provides stable per-example gradients (well-understood optimization landscape) while BPR provides explicit pairwise ranking signal by sampling (positive, negative) pairs from the same user group. The combination gives both calibrated probability estimates and correct relative ordering.

**Inference change:** XGBoost prediction traverses 175 independent decision trees per candidate, with each traversal involving 8 conditional branches (max_depth=8) -- these are sequential if/else operations that cannot be batched across trees on modern hardware. DCN-v2 prediction is 6 matrix multiplications that process all 200 candidates simultaneously in a single batched forward pass, achieving better hardware utilization on modern CPUs with SIMD/AVX instructions.

**Statistical evaluation upgrade:** The baseline project computed mean NDCG@10 for each model and declared the higher value "better." This ignores sampling variability entirely -- with ~6,000 test users, even identical models would show mean differences of +/-0.002 due to random test set composition. Our upgraded framework uses paired Wilcoxon signed-rank tests (comparing both rankers on the same user), 10,000-resample bootstrap confidence intervals, Benjamini-Hochberg FDR correction for the 6 metrics tested simultaneously, and Cohen's d effect sizes to distinguish statistical significance from practical significance.

---

*This notebook is a reference document. No code executes here. Proceed to Notebook 01 for EDA (unchanged) or Notebook 04 for DCN-v2 training.*